# Phase 2: Spending Pattern Analysis & Financial KPIs

**Days 10-18**: EDA, KPI Calculations, and Statistical Analysis

## CELL 1: Setup & Imports

**⚠️ CHANGE THIS CELL COMPLETELY**

Replace everything with code below:

In [ ]:
import pandas as pd
import numpy as np
import sys
from sqlalchemy import create_engine

# Add src to Python path
sys.path.insert(0, '../src')

# Import config
from pfm.config import DB_PATH, USERS, NEEDS_CATEGORIES, WANTS_CATEGORIES, SAVINGS_CATEGORIES

# Import spending_analysis functions
from pfm.analytics.spending_analysis import (
    spending_by_category,
    spending_by_merchant,
    spending_by_month,
    spending_by_user,
    spending_by_day_of_week,
    top_expense_drivers,
    weekend_vs_weekday_spending
)

# Import kpi_engine functions
from pfm.analytics.kpi_engine import (
    savings_rate,
    debt_to_income_ratio,
    budget_variance,
    emergency_fund_coverage,
    spending_50_30_20
)

# Create database connection
engine = create_engine(f'sqlite:///{DB_PATH}')

print('✅ Environment Ready')
print(f'Database: {DB_PATH}')

## CELL 2: Load Data

**Copy this cell as-is. No changes needed.**

In [ ]:
# Load transactions (expenses only)
df = pd.read_sql('SELECT * FROM transactions WHERE is_income = 0', engine)
df['date'] = pd.to_datetime(df['date'])

# Load budgets
budgets = pd.read_sql('SELECT * FROM budgets', engine)

print(f'\n📊 DATASET OVERVIEW')
print(f'Total transactions: {len(df):,}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Users: {df["user_name"].nunique()}')
print(f'Categories: {df["category"].nunique()}')
print(f'Total spend: Rs. {df["amount"].sum():,.0f}')
print(f'Avg transaction: Rs. {df["amount"].mean():,.2f}')

## CELL 3: Spending by Category

**Copy this cell as-is. No changes needed.**

In [ ]:
cat_spend = spending_by_category(df)

print(f'\n📊 TOP 10 CATEGORIES BY SPEND')
print(cat_spend.head(10))

## CELL 4: Top Merchants

**Copy this cell as-is. No changes needed.**

In [ ]:
merch = spending_by_merchant(df, top_n=10)

print(f'\n🏪 TOP 10 MERCHANTS')
print(merch.head(10))

## CELL 5: Monthly Spending Trend

**Copy this cell as-is. No changes needed.**

In [ ]:
monthly = spending_by_month(df)

print(f'\n📈 MONTHLY SPENDING TREND')
print(monthly)

## CELL 6: Spending by User

**Copy this cell as-is. No changes needed.**

In [ ]:
user_spend = spending_by_user(df)

print(f'\n👤 SPENDING BY USER')
print(user_spend)

## CELL 7: Financial KPIs by User

**Copy this cell as-is. No changes needed.**

In [ ]:
print(f'\n' + '='*70)
print('💰 FINANCIAL KPIs BY USER')
print('='*70)

for user_info in USERS:
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    print(f'\n{user_name}')
    print('-'*70)
    
    # Savings Rate
    sr = savings_rate(df, user_id)
    print(f'  Savings Rate: {sr["rate"]:.1f}% (Income: Rs. {sr["income"]:,.0f} | Expense: Rs. {sr["expenses"]:,.0f})')
    
    # Debt-to-Income Ratio
    dti = debt_to_income_ratio(df, user_id)
    print(f'  Debt-to-Income: {dti["dti"]:.1f}% (Monthly debt: Rs. {dti["monthly_debt"]:,.0f})')
    
    # Emergency Fund
    ef = emergency_fund_coverage(df, user_id, months=3)
    print(f'  Emergency Fund Coverage: {ef["coverage_months"]:.1f}x months (Target: Rs. {ef["target_emergency_fund"]:,.0f})')
    
    # 50/30/20 Rule
    rule = spending_50_30_20(df, user_id, NEEDS_CATEGORIES, WANTS_CATEGORIES, SAVINGS_CATEGORIES)
    print(f'  50/30/20 Rule: Needs {rule["needs_pct"]:.0f}% | Wants {rule["wants_pct"]:.0f}% | Savings {rule["savings_pct"]:.0f}%')

## CELL 8: Pareto Analysis (Top Expense Drivers)

**Copy this cell as-is. No changes needed.**

In [ ]:
top_cat, cum_pct = top_expense_drivers(df, top_n=10)

print(f'\n🎯 PARETO ANALYSIS (80/20 Rule)')
print(f'Top 10 categories account for {cum_pct:.1f}% of total spend')
print(f'\nTop categories:')
print(top_cat)

## CELL 9: Weekend vs Weekday

**Copy this cell as-is. No changes needed.**

In [ ]:
wknd = weekend_vs_weekday_spending(df)

print(f'\n📅 WEEKEND VS WEEKDAY SPENDING')
print(f'Weekday avg: Rs. {wknd[("amount", "mean")][False]:,.0f}')
print(f'Weekend avg: Rs. {wknd[("amount", "mean")][True]:,.0f}')
diff = ((wknd[("amount", "mean")][True] - wknd[("amount", "mean")][False]) / wknd[("amount", "mean")][False] * 100)
print(f'Difference: {diff:+.1f}%')

## CELL 10: Key Insights & Summary

**Copy this cell as-is. No changes needed.**

In [ ]:
print('\n' + '='*70)
print('✅ PHASE 2 ANALYSIS COMPLETE')
print('='*70)

total_spend = df['amount'].sum()
top_cat_name = cat_spend.index[0]
top_cat_amt = cat_spend[('amount', 'sum')][0]
top_cat_pct = (top_cat_amt / total_spend * 100)

print(f'\n📊 KEY FINDINGS:')
print(f'  1. Total spend: Rs. {total_spend:,.0f}')
print(f'  2. Top category: {top_cat_name} ({top_cat_pct:.1f}% of spend)')
print(f'  3. Pareto rule: {cum_pct:.1f}% in top 10 categories')
print(f'  4. Weekend premium: {diff:+.1f}% vs weekday')
print(f'  5. Users: {df["user_name"].nunique()}')

print(f'\n✅ Ready for Phase 3: Machine Learning Models')
print('='*70)